In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev_pnl",
    choices=["fq_dev_pnl", "fq_test_pnl", "fq_prod_pnl"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="other",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="dim_location_master",
    choices=["dim_location_master"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `fq_dev_extloc_staging`"
).select("url").collect()[0][0]

In [0]:
from pyspark.sql.functions import *

cdc_raw_data = spark.read.option('header', True).format('csv').load(f'{staging}/master_data/P&L Master/location_master_v3.csv').limit(1)
display(cdc_raw_data)

In [0]:
%sql CREATE EXTERNAL TABLE IF NOT EXISTS fq_dev_pnl_catalog.bronze.dim_location_master
USING DELTA
LOCATION 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/bronze/dim_location_master' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
import time
from pyspark.sql.functions import *
import re, time

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df

bronzeDF = (spark.read
                .format("csv")
                .option('header', True)
                .option("inferSchema", "true")
                .load(f'{staging}/master_data/P&L Master/location_master_v3.csv')
            )
# display(bronzeDF)
df = to_snake_case_df(bronzeDF)

(df.withColumn("ingestion_ts", current_timestamp())
        .withColumn('file_name', substring_index(col('_metadata.file_name'), '.', 1))
        .withColumn('file_path', regexp_replace(col('_metadata.file_path'), '%20', ' '))
        .withColumn('sys_id', expr('uuid()'))
        .write
        .mode('overwrite')
        .option('overwriteSchema', True)
        .saveAsTable(f"`{environment}_catalog`.`bronze`.`{domain}`", mergeSchema=True)
)


In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows
FROM fq_dev_pnl_catalog.bronze.dim_location_master;

In [0]:
%sql select * from fq_dev_pnl_catalog.bronze.dim_location_master limit 1